In [2]:
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
import os
openai_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [4]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [5]:
len(documents)

72

In [6]:
import minsearch

index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

query = "How does the agentic loop keep calling the model until it stops?"
search_results = index.search(query, num_results=1)

print(f"First result filename: {search_results[0]['filename']}")

First result filename: 01-agentic-rag/lessons/14-agentic-loop.md


In [13]:
import os
from rag_helper import RAGBase

class HomeworkRAG(RAGBase):
    
    def search(self, query, num_results=5):
        return self.index.search(
            query,
            num_results=num_results,
            boost_dict={'content': 1.0},
            filter_dict={}
        )

    def build_context(self, search_results):
        lines = []
        for doc in search_results:
            lines.append(f"Source file: {doc['filename']}")
            lines.append(doc['content'])
            lines.append("")
        return "\n".join(lines).strip()

    def llm(self, prompt):
        response = self.llm_client.chat.completions.create(
            model=self.model,
            messages=[
                {'role': 'developer', 'content': self.instructions},
                {'role': 'user', 'content': prompt}
            ]
        )
        return response.choices[0].message.content, response.usage.prompt_tokens

    def rag(self, query):
        search_results = self.search(query, num_results=5)
        prompt = self.build_prompt(query, search_results)
        return self.llm(prompt)

# Instantiate and execute
homework_assistant = HomeworkRAG(
    index=index,
    llm_client=openai_client,
    model="gemini-2.5-flash"
)

query = "How does the agentic loop keep calling the model until it stops?"
answer, input_tokens = homework_assistant.rag(query)

print(f"Total Input Tokens: {input_tokens}")

Total Input Tokens: 7953


In [14]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

print(f"Total chunks created: {len(chunks)}")

Total chunks created: 295


In [16]:
from gitsource import chunk_documents
import minsearch
chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Total Chunks: {len(chunks)}")

Total Chunks: 295


In [18]:
chunk_index = minsearch.Index(text_fields=["content"], keyword_fields=["filename"])
chunk_index.fit(chunks)
chunk_assistant = HomeworkRAG(
    index=chunk_index,
    llm_client=openai_client,
    model="gemini-2.5-flash"
)
_, chunked_tokens = chunk_assistant.rag(query)
print(f"Chunked Input Tokens: {chunked_tokens}")

Chunked Input Tokens: 2605


In [21]:
import json

def run_agent_search(query):
    results = chunk_index.search(query, num_results=5)
    return [doc['content'] for doc in results]

agent_messages = [
    {"role": "developer", "content": "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."},
    {"role": "user", "content": "How does the agentic loop work, and how is it different from plain RAG?"}
]

search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the course lessons chunk database for specific technical keywords.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string"}
            },
            "required": ["query"]
        }
    }
}

search_counter = 0

for _ in range(10):
    response = openai_client.chat.completions.create(
        model="gemini-2.5-flash",
        messages=agent_messages,
        tools=[search_tool]
    )
    
    assistant_message = response.choices[0].message
    
    if assistant_message.tool_calls:
        agent_messages.append(assistant_message)
        tool_call = assistant_message.tool_calls[0]
        search_query = json.loads(tool_call.function.arguments).get("query")
        
        search_counter += 1
        search_results = run_agent_search(search_query)
        
        agent_messages.append({
            "role": "tool",
            "name": "search",
            "tool_call_id": tool_call.id,
            "content": json.dumps(search_results)
        })
    else:
        print(assistant_message.content)
        break

print(f"\nTotal searches: {search_counter}")

RateLimitError: Error code: 429 - [{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 5.737814804s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'}, 'q